# (DSC527) Interactive Local Dashboard with Simulated Real-Time Data – Part 1
Author: Tracey Johnson/
Date: 08-05-2026/
Python version 3.13

---

## Data Setup

Import libraries.

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print("Libraries imported successfully.")

Libraries imported successfully.


The synthetic dataset was generated in Python using configurable statistical distributions for each feature, with optional noise and missingness to mimic real-world variability. Features were cast into stable types—including categorical labels, integers, and floats—to ensure a consistent schema. The generator supports append-ready behavior by producing sequential IDs and timestamps, allowing new rows to be added later without regenerating the full dataset. This design makes the dataset scalable, reproducible, and suitable for periodic expansion.

`Generate_synthetic_distribution`, function generates synthetic data based on specified distribution types (uniform, normal, exponential) and parameters.

In [13]:
def _generate_base_distribution(distribution, dist_params, num_samples):
    distribution = distribution.lower()

    if distribution == 'normal':
        loc = dist_params.get('loc', 0.0)
        scale = dist_params.get('scale', 1.0)
        return np.random.normal(loc=loc, scale=scale, size=num_samples)

    elif distribution == 'uniform':
        low = dist_params.get('low', 0.0)
        high = dist_params.get('high', 1.0)
        return np.random.uniform(low=low, high=high, size=num_samples)

    elif distribution == 'exponential':
        scale = dist_params.get('scale', 1.0)
        return np.random.exponential(scale=scale, size=num_samples)

    elif distribution == 'logistic':
        loc = dist_params.get('loc', 0.0)
        scale = dist_params.get('scale', 1.0)
        return np.random.logistic(loc=loc, scale=scale, size=num_samples)

    elif distribution == 'beta':
        a = dist_params.get('a', 2.0)
        b = dist_params.get('b', 5.0)
        return np.random.beta(a, b, size=num_samples)

    else:
        raise ValueError(f"Unsupported distribution: {distribution}")

`Convert_to_type`, function converts a numpy array to the specified data type, handling float, int, and categorical types appropriately.

In [14]:
def _convert_to_type(data_array, data_type):
    data_type = data_type.lower()

    if data_type == 'float':
        return data_array.astype(float)

    elif data_type == 'int':
        if np.isnan(data_array).any():
            return data_array.astype(float)
        else:
            return np.round(data_array).astype(int)

    elif data_type == 'categorical':
        clean_int = np.round(data_array).astype(int)
        return pd.Categorical(clean_int)

    else:
        raise ValueError(f"Unsupported data_type: {data_type}")

The main function, `generate_synthetic_dataset`, creates a synthetic dataset based on the provided configuration. It allows for customization of the number of samples, feature configurations, and random seed for reproducibility.

In [15]:
def generate_synthetic_dataset(
    num_samples=1000,
    features_config=None,
    random_seed=None,
    start_id=None,
    add_timestamp=True
):
    if random_seed is not None:
        np.random.seed(random_seed)

    if features_config is None or not isinstance(features_config, list):
        raise ValueError("features_config must be a list.")

    df = pd.DataFrame()

    # append‑ready ID
    if start_id is not None:
        df["id"] = np.arange(start_id, start_id + num_samples)

    # generate all feature columns
    for feature in features_config:
        col_name = feature.get('name')
        distribution = feature.get('distribution')
        dist_params = feature.get('dist_params', {})
        data_type = feature.get('data_type', 'float')
        noise_std = feature.get('noise_std', 0.0)
        missing_rate = feature.get('missing_rate', 0.0)
        labels = feature.get('categories')

        feature_seed = feature.get('random_seed')
        if feature_seed is not None:
            np.random.seed(feature_seed)

        col_data = _generate_base_distribution(distribution, dist_params, num_samples)

        if feature_seed is not None and random_seed is not None:
            np.random.seed(random_seed)

        if noise_std > 0:
            col_data += np.random.normal(0, noise_std, num_samples)

        if missing_rate > 0:
            n_missing = int(missing_rate * num_samples)
            idx = np.random.choice(num_samples, n_missing, replace=False)
            col_data[idx] = np.nan

        if data_type == 'int':
            col_data = col_data.astype(float) if np.isnan(col_data).any() else np.round(col_data).astype(int)

        elif data_type == 'float':
            col_data = col_data.astype(float)

        elif data_type == 'categorical':
            col_data = np.round(col_data).astype(int)
            if labels:
                col_data = np.clip(col_data, 0, len(labels) - 1)
                col_data = pd.Categorical([labels[i] for i in col_data])
            else:
                col_data = pd.Categorical(col_data)

        df[col_name] = col_data

    # add timestamp
    if add_timestamp:
        df["time_stamp"] = pd.Timestamp.now()

    return df

Define the feature configuration for the synthetic dataset, specifying the desired distributions and data types for each feature.

In [16]:
gaming_features = [
    {
        'name': 'game_genre',
        'distribution': 'uniform',
        'dist_params': {'low': 0, 'high': 5},
        'data_type': 'categorical',
        'categories': ["Shooter", "RPG", "Action", "Adventure", "Fighting"]
    },
    {
        'name': 'game_theme',
        'distribution': 'uniform',
        'dist_params': {'low': 0, 'high': 3},
        'data_type': 'categorical',
        'categories': ["Fantasy", "Crime", "Sports"]
    },
    {
        'name': 'player_perspective',
        'distribution': 'uniform',
        'dist_params': {'low': 0, 'high': 2},
        'data_type': 'categorical',
        'categories': ["First-Person", "Third-Person"]
    },
    {
        'name': 'game_mode',
        'distribution': 'uniform',
        'dist_params': {'low': 0, 'high': 3},
        'data_type': 'categorical',
        'categories': ["Single-Player", "Multiplayer", "Battle Royale"]
    },
    {
        'name': 'age_rating',
        'distribution': 'uniform',
        'dist_params': {'low': 10, 'high': 19},
        'data_type': 'int'
    },
    {
        'name': 'supported_platforms',
        'distribution': 'normal',
        'dist_params': {'loc': 3, 'scale': 1},
        'data_type': 'int'
    },
    {
        'name': 'release_year',
        'distribution': 'normal',
        'dist_params': {'loc': 2018, 'scale': 3},
        'data_type': 'int'
    },
    {
        'name': 'rating_score',
        'distribution': 'normal',
        'dist_params': {'loc': 75, 'scale': 10},
        'data_type': 'float'
    },
    {
        'name': 'rating_count',
        'distribution': 'exponential',
        'dist_params': {'scale': 500},
        'data_type': 'int'
    }
]

df_syn = generate_synthetic_dataset(
    num_samples=1000000,
    features_config=gaming_features,
    random_seed=42
)

Finally, add realistic variation to the synthetic dataset. 

In [17]:
df_syn.loc[df_syn['game_genre'] == "RPG", 'rating_score'] += 5
df_syn.loc[df_syn['game_theme'] == "Sports", 'rating_score'] -= 7
mask = df_syn['game_mode'] == "Battle Royale"
df_syn.loc[mask, 'rating_score'] += np.random.normal(0, 8, mask.sum())
df_syn.loc[df_syn['player_perspective'] == "First-Person", 'rating_score'] -= 3
df_syn['rating_score'] = df_syn['rating_score'].clip(0, 100)

In [18]:
print(df_syn.head())
print(df_syn.tail())

  game_genre game_theme player_perspective      game_mode  age_rating  \
0     Action     Sports       First-Person  Battle Royale          17   
1   Fighting      Crime       Third-Person    Multiplayer          12   
2   Fighting    Fantasy       Third-Person  Battle Royale          10   
3  Adventure     Sports       Third-Person  Single-Player          10   
4        RPG     Sports       First-Person  Battle Royale          18   

   supported_platforms  release_year  rating_score  rating_count  \
0                    4          2017     57.411678           664   
1                    3          2017     64.803982           631   
2                    5          2020     78.862346            18   
3                    2          2019     62.072248          1056   
4                    5          2019     68.220565           221   

                  time_stamp  
0 2026-08-05 22:07:19.062288  
1 2026-08-05 22:07:19.062288  
2 2026-08-05 22:07:19.062288  
3 2026-08-05 22:07:19.062288

Save the synthetic dataset to a CSV file for later use in the dashboard.

In [21]:
df_syn.to_csv('synthetic_gaming_data.csv', index=False)

---

## Exploratory Analysis

In [ ]:
print("Synthetic dataset generated successfully.")
print(df_syn.info())

Synthetic dataset generated successfully.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 10 columns):
 #   Column               Non-Null Count    Dtype         
---  ------               --------------    -----         
 0   game_genre           1000000 non-null  category      
 1   game_theme           1000000 non-null  category      
 2   player_perspective   1000000 non-null  category      
 3   game_mode            1000000 non-null  category      
 4   age_rating           1000000 non-null  int64         
 5   supported_platforms  1000000 non-null  int64         
 6   release_year         1000000 non-null  int64         
 7   rating_score         1000000 non-null  float64       
 8   rating_count         1000000 non-null  int64         
 9   time_stamp           1000000 non-null  datetime64[us]
dtypes: category(4), datetime64[us](1), float64(1), int64(4)
memory usage: 49.6 MB
None


### Univariate Analysis

In [ ]:
print("Descriptive statistics for categorical features:")
print(df_syn.describe(include=['category']))

Descriptive statistics for categorical features:
       game_genre game_theme player_perspective      game_mode
count     1000000    1000000            1000000        1000000
unique          5          3                  2              3
top      Fighting     Sports       Third-Person  Battle Royale
freq       300210     498981             750042         499760


In [ ]:
for col in df_syn.select_dtypes(include=['category']).columns:
    plt.figure(figsize=(6, 4))
    counts = df_syn[col].value_counts()
    plt.bar(counts.index.astype(str), counts.values, color="steelblue", alpha=0.8)
    plt.title(col, fontsize=12)
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
print("Descriptive statistics for numerical features:")
print(df_syn.describe(include=[np.number]))

In [ ]:
for col in df_syn.select_dtypes(include=[np.number]).columns:
    plt.figure(figsize=(6, 4))
    plt.hist(df_syn[col].dropna(), color="steelblue", alpha=0.8)
    plt.title(col, fontsize=12)
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.grid(axis='y', alpha=0.75)
    plt.tight_layout()
    plt.show()

### Bivaraiate Analysis

In [ ]:
for col in df_syn.select_dtypes(include=['category']).columns:
    df_syn.boxplot(column='rating_score', by=col)
    plt.title(f'rating_score by {col}')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()

### Multivariate Analysis

In [20]:
print("Correlation matrix for numerical features:")
print(df_syn.corr(numeric_only=True))

Correlation matrix for numerical features:
                     age_rating  supported_platforms  release_year  \
age_rating             1.000000            -0.000881     -0.000978   
supported_platforms   -0.000881             1.000000      0.000085   
release_year          -0.000978             0.000085      1.000000   
rating_score          -0.000221            -0.000299      0.000500   
rating_count          -0.000942             0.000037      0.001560   

                     rating_score  rating_count  
age_rating              -0.000221     -0.000942  
supported_platforms     -0.000299      0.000037  
release_year             0.000500      0.001560  
rating_score             1.000000      0.000616  
rating_count             0.000616      1.000000  


---

## References

Anoop, P. M. (2023, May 31). A powerful duo for data visualization: Streamlit and Plotly. Medium. https://medium.com/tech-blogs-by-nest-digital/a-powerful-duo-for-data-visualization-streamlit-and-plotly-58894dee4b6d

Hunicke, R., LeBlanc, M., & Zubek, R. (2004). MDA: A formal approach to game design and game research. Game Developers Conference. https://users.cs.northwestern.edu/~hunicke/MDA.pdf

Plotly Technologies Inc. (2026). How do Dash, Posit (Shiny), Streamlit, and Bokeh compare as low‑code UI layers for AI and ML models? Plotly. https://plotly.com/compare-dash-shiny-streamlit-bokeh

Pyrcz, M. J. (2024). Applied geostatistics in Python: A hands‑on guide with GeostatsPy. https://geostatsguy.github.io/GeostatsPyDemos_Book/GeostatsPy_simulation_postsim.html

Streamlit. (2026). Streamlit documentation. https://docs.streamlit.io

Zhang, Z., Wen, J., Chen, Z., Arbab, D., Sahani, S., Arbab, B., Jin, H., & Rahman, T. (2024). Predicting quality of video gaming experience using global-scale telemetry data and federated learning. arXiv. https://arxiv.org/abs/2412.08950